# PoliMillionaire baseline

Before running this notebook, put the files in Google Drive like this:

```
MyDrive/
└── Colab Notebooks/
    └── NLP_assignment/
        ├── poli_millionaire_clean_baseline_v2.ipynb
        └── millionaire_client/
            ├── __init__.py
            ├── client.py
            ├── auth.py
            ├── base.py
            ├── game.py
            ├── models.py
            ├── competitions.py
            ├── leaderboard.py
            └── exceptions.py
```

In [1]:
from google.colab import drive
drive.mount('/content/gdrive/')

import os
import sys
import time
import re
import json
import math
import itertools
import torch

Drive already mounted at /content/gdrive/; to attempt to forcibly remount, call drive.mount("/content/gdrive/", force_remount=True).


In [2]:
BASE_DIR = '/content/gdrive/MyDrive/NLP_assignment1'
PACKAGE_DIR = os.path.join(BASE_DIR, 'millionaire_client')

print('BASE_DIR exists:', os.path.exists(BASE_DIR))
if os.path.exists(BASE_DIR):
    print('BASE_DIR contents:', os.listdir(BASE_DIR))

print('PACKAGE_DIR exists:', os.path.exists(PACKAGE_DIR))
if os.path.exists(PACKAGE_DIR):
    print('PACKAGE_DIR contents:', os.listdir(PACKAGE_DIR))

if not os.path.exists(BASE_DIR):
    raise FileNotFoundError('BASE_DIR not found. Create the NLP_assignment folder in Drive and upload the notebook there.')

if not os.path.exists(PACKAGE_DIR):
    raise FileNotFoundError('millionaire_client folder not found inside BASE_DIR.')

if BASE_DIR not in sys.path:
    sys.path.append(BASE_DIR)

print('Path added successfully.')

BASE_DIR exists: True
BASE_DIR contents: ['PoliMillionaire.ipynb', 'millionaire_client', '.ipynb_checkpoints', 'test3_simple_rag_game_runs']
PACKAGE_DIR exists: True
PACKAGE_DIR contents: ['base.py', 'game.py', 'client.py', 'leaderboard.py', 'competitions.py', 'exceptions.py', '__init__.py', 'models.py', 'auth.py', '__pycache__']
Path added successfully.


In [3]:
%pip install -q langchain transformers accelerate sentencepiece protobuf sympy
%pip install -q -U 'bitsandbytes>=0.46.1'

In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from langchain_core.language_models.llms import LLM
from langchain_core.prompts import PromptTemplate
import sympy as sp
from millionaire_client import MillionaireClient
from millionaire_client.exceptions import TimeoutError, RateLimitError

In [5]:
API_URL = 'http://131.175.15.22:51111/'
USERNAME = 'gary'
PASSWORD = '13790229'

client = MillionaireClient(API_URL)
user = client.login(USERNAME, PASSWORD)
print('Logged in as:', user.username)

Logged in as: gary


In [ ]:
competitions = client.competitions.list_all()
for c in competitions:
    print(c.id, c.name, c.max_levels)

COMPETITION_ID = competitions[3].id

0 Entertainment 15
1 Ancient History and Politics 15
2 Science and Nature 15
3 Maths 15


In [7]:
model_id = 'deepseek-ai/DeepSeek-R1-Distill-Qwen-7B'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map='auto'
)
model.eval()

class LocalModelLLM(LLM):
    max_new_tokens: int = 1

    @property
    def _llm_type(self):
        return 'local_deepseek_r1_distill_qwen'

    def _call(self, prompt, stop=None, run_manager=None, **kwargs):
        inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
        input_len = inputs['input_ids'].shape[1]

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=self.max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )

        text = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)
        if stop:
            for marker in stop:
                text = text.split(marker, 1)[0]
        return text

llm = LocalModelLLM(max_new_tokens=4)
calculator_llm = LocalModelLLM(max_new_tokens=160)
reasoning_llm = LocalModelLLM(max_new_tokens=160)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

In [8]:
calculator_request_prompt = PromptTemplate.from_template(
    '''Decide whether a calculator helps answer this multiple choice question.
Return only one JSON object. Do not use Markdown or explanation text.
Use a calculator for equations, exact arithmetic, ones digits, LCM recurrence problems, finite die-roll product probabilities, normal probabilities or percentiles, and triangle areas from three altitudes.
Skip it for vocabulary, definitions, names, and conceptual statistics questions.

Supported JSON shapes:
Skip: {{"use_calculator": false, "kind": "skip", "reason": "conceptual"}}
Equation: {{"use_calculator": true, "kind": "equation", "equation": "(1/25)**(x+2) = 125**(-x)", "variable": "x"}}
Expression: {{"use_calculator": true, "kind": "expression", "expression": "(18 + 6) / 4"}}
Ones digit: {{"use_calculator": true, "kind": "ones_digit", "expression": "1*2*3*4*5*6*7*8*9"}}
LCM: {{"use_calculator": true, "kind": "lcm", "values": [360, 450, 540]}}
Roll product probability: {{"use_calculator": true, "kind": "product_multiple_probability", "roll_ranges": [[1, 8], [1, 6]], "multiple": 3}}
Normal percentile: {{"use_calculator": true, "kind": "normal_cdf", "value": 90, "mean": 80, "std_dev": 9, "top_percent": 15}}
Normal upper tail: {{"use_calculator": true, "kind": "normal_probability", "value": 3000, "mean": 2500, "std_dev": 225, "tail": "upper"}}
Altitude area: {{"use_calculator": true, "kind": "triangle_area_from_altitudes", "altitudes": [10, 12, 15]}}

Write arithmetic as plain Python-style math. Use ** for powers, * for multiplication, sqrt(...) for roots, and no LaTeX.

Question: {question}
A) {option_a}
B) {option_b}
C) {option_c}
D) {option_d}

JSON:'''
)

reasoning_prompt = PromptTemplate.from_template(
    '''Solve this PoliMillionaire multiple choice question carefully before the final answer picker runs.
Use the calculator result JSON when it has used=true and no error.
If the calculator was skipped or errored, reason from the question and options.

Return only one JSON object. Do not use Markdown or text outside JSON.
JSON format:
{{"best_option": "A", "short_reason": "brief decisive reason", "confidence": "low"}}

Rules:
- best_option must be exactly A, B, C, or D.
- short_reason must be at most one short sentence.
- confidence must be exactly low, medium, or high.

Question: {question}
A) {option_a}
B) {option_b}
C) {option_c}
D) {option_d}

Calculator result JSON: {calculator_result}

JSON:'''
)

answer_prompt = PromptTemplate.from_template(
    '''PoliMillionaire MCQ. Reply only with A, B, C, or D.
Use the calculator result JSON when it has used=true and no error.
Use the reasoning result JSON when it has a best_option.
Choose the option that matches the solved result. If the calculator was skipped or errored, use the reasoning result and question.

Question: {question}
A) {option_a}
B) {option_b}
C) {option_c}
D) {option_d}

Calculator result JSON: {calculator_result}
Reasoning result JSON: {reasoning_result}

Answer:'''
)

calculator_request_chain = calculator_request_prompt | calculator_llm
reasoning_chain = reasoning_prompt | reasoning_llm
answer_chain = answer_prompt | llm

def solve_calculator_request(raw_request):
    """Solve one calculator request and return a JSON result."""
    output = {'used': False, 'request': None}

    try:
        if isinstance(raw_request, dict):
            request = raw_request
        else:
            raw_text = str(raw_request).strip()
            json_start = raw_text.find('{')
            request, _ = json.JSONDecoder().raw_decode(raw_text[json_start:] if json_start >= 0 else raw_text)

        output['request'] = request
        kind = str(request.get('kind', 'skip')).strip()
        output['kind'] = kind

        if not request.get('use_calculator') or kind == 'skip':
            output['result'] = {'message': 'No calculator needed.'}
            return json.dumps(output, ensure_ascii=True)

        output['used'] = True
        safe_locals = {
            'sqrt': sp.sqrt,
            'factorial': sp.factorial,
            'pi': sp.pi,
            'E': sp.E
        }

        def normalize_expression(expression):
            expression = str(expression).replace('^', '**')
            expression = expression.replace('{', '(').replace('}', ')')
            if chr(92) in expression or '__' in expression:
                raise ValueError('Calculator expressions must not contain LaTeX or private names.')
            if not re.fullmatch(r'[0-9A-Za-z_+\-*/().,%=\s]+', expression):
                raise ValueError('Calculator expression has unsupported characters.')
            return expression

        if kind == 'expression':
            expression = normalize_expression(request['expression'])
            value = sp.simplify(sp.sympify(expression, locals=safe_locals))
            if value.free_symbols:
                raise ValueError('Expression still contains variables.')
            output['result'] = {'exact': str(value), 'decimal': str(sp.N(value, 10))}

        elif kind == 'ones_digit':
            expression = normalize_expression(request['expression'])
            value = sp.simplify(sp.sympify(expression, locals=safe_locals))
            if value.free_symbols:
                raise ValueError('Ones digit expression still contains variables.')
            output['result'] = {'integer': str(value), 'ones_digit': int(value) % 10}

        elif kind == 'lcm':
            values = [int(value) for value in request['values']]
            if not values or any(value <= 0 for value in values):
                raise ValueError('LCM requests need positive integer values.')
            value = math.lcm(*values)
            output['result'] = {'values': values, 'lcm': value}

        elif kind == 'product_multiple_probability':
            multiple = int(request['multiple'])
            if multiple <= 0:
                raise ValueError('Product multiple must be positive.')
            roll_ranges = []
            for roll_range in request['roll_ranges']:
                if len(roll_range) != 2:
                    raise ValueError('Each roll range needs a first and last value.')
                first, last = [int(value) for value in roll_range]
                if first > last:
                    raise ValueError('Roll range lower bound cannot exceed upper bound.')
                roll_ranges.append(range(first, last + 1))
            if not roll_ranges:
                raise ValueError('At least one roll range is required.')
            outcomes = list(itertools.product(*roll_ranges))
            favorable = sum(math.prod(outcome) % multiple == 0 for outcome in outcomes)
            probability = sp.Rational(favorable, len(outcomes))
            output['result'] = {
                'favorable_outcomes': favorable,
                'total_outcomes': len(outcomes),
                'probability': str(probability),
                'decimal': str(sp.N(probability, 10))
            }

        elif kind == 'equation':
            variable_name = str(request.get('variable', 'x')).strip()
            if not re.fullmatch(r'[A-Za-z][A-Za-z0-9_]*', variable_name):
                raise ValueError('Equation variable name is invalid.')
            variable = sp.Symbol(variable_name)
            safe_locals[variable_name] = variable
            equation = normalize_expression(request['equation'])
            if equation.count('=') != 1:
                raise ValueError('Equation requests need exactly one equals sign.')
            left, right = equation.split('=', 1)
            solutions = sp.solve(
                sp.Eq(sp.sympify(left, locals=safe_locals), sp.sympify(right, locals=safe_locals)),
                variable
            )
            output['result'] = {
                'variable': variable_name,
                'solutions': [str(sp.simplify(solution)) for solution in solutions],
                'decimal_solutions': [str(sp.N(solution, 10)) for solution in solutions]
            }

        elif kind == 'normal_cdf':
            value = float(request['value'])
            mean = float(request['mean'])
            std_dev = float(request['std_dev'])
            if std_dev <= 0:
                raise ValueError('Standard deviation must be positive.')
            z_score = (value - mean) / std_dev
            z_for_table = round(z_score, int(request.get('z_round_digits', 2)))
            percentile = 100 * 0.5 * (1 + math.erf(z_for_table / math.sqrt(2)))
            result = {
                'z_score': round(z_score, 6),
                'z_score_for_percentile': z_for_table,
                'percentile': round(percentile, 2)
            }
            if request.get('top_percent') is not None:
                result['qualifies_top_percent'] = percentile >= 100 - float(request['top_percent'])
            output['result'] = result

        elif kind == 'normal_probability':
            value = float(request['value'])
            mean = float(request['mean'])
            std_dev = float(request['std_dev'])
            tail = str(request.get('tail', 'lower')).strip().lower()
            if std_dev <= 0:
                raise ValueError('Standard deviation must be positive.')
            if tail not in {'lower', 'upper'}:
                raise ValueError('Normal probability tail must be lower or upper.')
            z_score = (value - mean) / std_dev
            z_for_table = round(z_score, int(request.get('z_round_digits', 2)))
            lower_probability = 0.5 * (1 + math.erf(z_for_table / math.sqrt(2)))
            probability = lower_probability if tail == 'lower' else 1 - lower_probability
            output['result'] = {
                'tail': tail,
                'z_score': round(z_score, 6),
                'z_score_for_probability': z_for_table,
                'probability': round(probability, 4)
            }

        elif kind == 'triangle_area_from_altitudes':
            altitudes = [sp.Rational(str(value)) for value in request['altitudes']]
            if len(altitudes) != 3 or any(value <= 0 for value in altitudes):
                raise ValueError('Triangle altitude requests need three positive values.')
            side_ratios = [1 / value for value in altitudes]
            semiperimeter = sum(side_ratios) / 2
            ratio_area_squared = semiperimeter
            for side_ratio in side_ratios:
                ratio_area_squared *= semiperimeter - side_ratio
            if ratio_area_squared <= 0:
                raise ValueError('The altitudes do not describe a triangle.')
            area = sp.simplify(1 / (4 * sp.sqrt(ratio_area_squared)))
            output['result'] = {'exact': str(area), 'decimal': str(sp.N(area, 10))}

        else:
            raise ValueError(f'Unsupported calculator kind: {kind}')

    except Exception as exc:
        output['error'] = f'{type(exc).__name__}: {exc}'

    return json.dumps(output, ensure_ascii=True)

# Add future tool functions next to this calculator function.
answer_tools = [solve_calculator_request]

def parse_json_object(raw_text):
    raw_text = str(raw_text).strip()
    json_start = raw_text.find('{')
    if json_start < 0:
        return None
    try:
        value, _ = json.JSONDecoder().raw_decode(raw_text[json_start:])
    except json.JSONDecodeError:
        return None
    return value if isinstance(value, dict) else None

def calculator_solved(calculator_result):
    result = parse_json_object(calculator_result)
    return bool(result and result.get('used') and not result.get('error'))

def extract_reasoning_letter(reasoning_result):
    result = parse_json_object(reasoning_result)
    best_option = str(result.get('best_option', '')).strip().upper() if result else ''
    return best_option if best_option in {'A', 'B', 'C', 'D'} else None

def extract_letter(text):
    text = text.strip().upper()
    match = re.search(r'\b([ABCD])\b', text)
    if match:
        return match.group(1)
    if text and text[0] in 'ABCD':
        return text[0]
    return 'A'

def choose_answer(question):
    if len(question.options) < 4:
        return question.options[0].id, 'A', 'fallback'

    inputs = {
        'question': question.text,
        'option_a': question.options[0].text,
        'option_b': question.options[1].text,
        'option_c': question.options[2].text,
        'option_d': question.options[3].text
    }
    calculator_request_json = calculator_request_chain.invoke(inputs)
    calculator_result = solve_calculator_request(calculator_request_json)
    reasoning_result_json = json.dumps({'skipped': 'calculator solved the request'})
    letter = None
    text = None

    if not calculator_solved(calculator_result):
        reasoning_result_json = reasoning_chain.invoke({**inputs, 'calculator_result': calculator_result})
        letter = extract_reasoning_letter(reasoning_result_json)
        if letter:
            text = 'reasoning.best_option'

    if not letter:
        text = answer_chain.invoke({
            **inputs,
            'calculator_result': calculator_result,
            'reasoning_result': reasoning_result_json
        })
        letter = extract_letter(text)
    idx = ['A', 'B', 'C', 'D'].index(letter)
    raw = {
        'calculator_request_json': calculator_request_json,
        'calculator_result': calculator_result,
        'reasoning_result_json': reasoning_result_json,
        'answer_output': text
    }
    return question.options[idx].id, letter, raw

In [9]:
def play_game(COMPETITION_ID=COMPETITION_ID):
    game = client.game.start(competition_id=COMPETITION_ID, mode='text')

    while game.in_progress:
        question = game.current_question
        if question is None:
            break

        print('Level:', game.current_level)
        print(question.text)
        for i, opt in enumerate(question.options):
            print(f"{chr(65+i)}) {opt.text}")

        option_id, letter, raw = choose_answer(question)
        print('Predicted:', letter, '| Raw output:', raw)

        try:
            result = game.answer(option_id)
        except TimeoutError:
            print('Timed out')
            break
        except RateLimitError:
            print('Rate limited, waiting...')
            time.sleep(5)
            result = game.answer(option_id)

        print('Correct:', result.correct, '| Earned:', result.earned_amount)

        if result.game_over:
            break

        time.sleep(0.1)

    return print('Final earned:', game.earned_amount)

In [18]:
play_game(3)

Level: 1
The probability that a point (x, y) in R^2 is chosen follows a uniform random distribution within the region described by the inequality 0 < |x| + |y| < 1 . What is the probability that 2(x + y) > 1?
A) 0
B) 1/sqrt(2)
C) sqrt(2)/4
D) 1/4
Predicted: A | Raw output: {'calculator_request_json': ' {"use_calculator": true, "kind": "product_multiple_probability", "roll_ranges": [[1, 8], [1, 6]], "multiple": 3}\n\nWait, no, that\'s not the question. The question is about a probability within a region defined by 0 < |x| + |y| < 1, and then the probability that 2(x + y) > 1.\n\nSo, first, I need to visualize the region where 0 < |x| + |y| < 1. This is a diamond or a square rotated by 45 degrees, centered at the origin, with vertices at (1,0), (0,1), (-1,0), and (0,-1). The area of this region', 'calculator_result': '{"used": true, "request": {"use_calculator": true, "kind": "product_multiple_probability", "roll_ranges": [[1, 8], [1, 6]], "multiple": 3}, "kind": "product_multiple_probab